# Scopus author profiles of Nobel laureates

This notebook finds the **Scopus author profile** of each Nobel laureate in an input list and retrieves the profile's metrics (documents, citations, h-index, co-authors, publication years, last known affiliation, subject areas). It is meant to be read: every rule used to pick a profile is explained below, together with the reason for it and the real cases that motivated it, so that anyone using the output table knows how much to trust each row and which rows to check by hand.

**Contents**

1. How it works, in short
2. Design decisions (why each rule exists)
3. `match_status` and the review flags
4. Output columns
5. How to review the output
6. Known limitations
7. Requirements and reproducibility
8. The code, step by step: configuration → input list → Wikidata → Scopus candidates → choice and checks → run → uncertain cases

## 1. How it works, in short

Wikidata tells us **who** each laureate is; Scopus is the source of truth for **which profile** is theirs and for every metric.

1. **Wikidata** – each name in the input list is matched to a person among the laureates of the *same prize category and award year*. Wikidata then gives that person's English name, alternative names, Nobel Prizes, birth and death years, employers and — when someone has recorded them — ORCID and Scopus Author ID.
2. **Scopus Author Search** – candidate profiles are collected with several searches (the Wikidata ORCID, the Wikidata name and aliases, and the name in the input list) and merged. Each candidate is checked against the laureate's names (surname and initials).
3. **Choice and checks** – the candidate that Wikidata points to (same Scopus ID or ORCID) is chosen; otherwise the name-compatible candidate with most documents. The chosen profile is retrieved with the **Author Retrieval API**, compared again with Wikidata, and given two sanity flags. A `match_status` summarises how confident the choice is.

Two files are written: one row per laureate (`output/nobel_scopus_profiles.csv`) and one row per candidate profile (`output/nobel_scopus_candidates.csv`), so every automatic choice can be checked against the alternatives it was chosen from.

## 2. Design decisions

Each decision below records **what** the notebook does, **why**, and, where relevant, the **alternative that was rejected**. The examples are cases met while developing the notebook on the 2000–2025 laureates.

**D1 – Two sources: Wikidata for identity, Scopus for the profile.**
A Scopus name search cannot know which "Robert Edwards" won a Nobel Prize: it returns every profile with that name (177 for Robert Edwards). Wikidata records the prize and its year, so it can tell *who* the laureate is; Scopus is still the only source for the profile and its metrics. A name-only version without Wikidata (`nobel_scopus_by_name.ipynb`) was built first and kept for comparison; it fails on transliterations, nicknames and homonyms (see D2, D3).

**D2 – Names are matched to Wikidata by prize category and year.**
Each input name is compared only with the one to three laureates of the same category and year. Within such a small group, approximate matching is safe, which handles transliterations (*Jores Ivanovitch Alfiorov* → Zhores Alferov), married or extended names (*Jennifer A. Doudna Cate* → Jennifer Doudna), nicknames (*Richard Timothy Hunt* → Tim Hunt) and homonyms (*Robert Butler Wilson Jr.*, Economics 2020, is never confused with Robert Woodrow Wilson, Physics 1978). Matching by name across all laureates is used only as a fallback, and flagged in `wd_match`.

**D3 – Scopus is searched with several name forms, using surname + first given name.**
Scopus profiles usually store a surname plus a first name or initials ("Higgs, P.W."), so full names with middle names often find nothing; the query is therefore `AUTHLAST(surname) AND AUTHFIRST(first given name)`. It is run for the Wikidata name, up to `MAX_ALIASES` Wikidata aliases and the input name, because a laureate may be indexed under any of them (*Tim Hunt* is found by "Tim", not by "Richard"). Fallbacks — first initial only, other given names (*Kamer **Daron** Acemoğlu*), reversed order for names written surname first (*Tu Youyou*) — run only if the main searches find no compatible candidate, to save requests.

**D4 – Candidates must match the laureate's surname and initials.**
- *Same surname*: Scopus sometimes returns profiles whose other names match the query — `AUTHLAST(Axel)` returned *Axel Richard **Stocker***. Such candidates are rejected.
- *First initial*: must be the initial of one of the laureate's given names (from all known name forms).
- *Extra initials*: only rule a candidate out when the laureate's middle names are known. *Richard **H.** Hunt*, a prolific namesake, is rejected for *Richard **Timothy** Hunt*; but *R. **A.** Axel* is accepted for *Richard Axel*, whose middle name is not recorded anywhere. An earlier, stricter version rejected the real Richard Axel and picked the 2-document Stocker profile instead.
- Multi-letter initials count as one (*Zh.* for Zhores).

**D5 – The choice: Wikidata first, then most documents.**
If a candidate has the Scopus ID or ORCID recorded in Wikidata, it is chosen. Otherwise the name-compatible candidate with most documents is chosen: laureates usually publish far more than their namesakes. Document counts come from the search itself, so all candidates are compared on the same basis at no extra cost.

**D6 – A Wikidata match is reported as `likely`, not `confirmed`.**
Wikidata is edited by volunteers; its Scopus IDs can be outdated (Scopus merges duplicate profiles) or point to a minor duplicate. For *Robert Geoffrey Edwards*, the Scopus ID in Wikidata is a 5-document profile with publications from 1973–1995 at another institution, while larger compatible profiles exist; with 177 namesakes, a wrong ID easily coincides with one of the candidates. The project decided to keep trusting Wikidata for the choice, but to call the result `likely` and to expose flags (D7) so such rows can be spotted.

**D7 – Doubtful choices are flagged, not automatically overridden.**
Two flags are computed for every chosen profile: `most_documents` (is it the largest name-compatible candidate?) and `award_in_scopus_pub_range` (do its publications cover the award year?). An automatic override — e.g. "replace the choice by any candidate with 3× more documents" — was considered and rejected: it would undo correct choices (Tim Hunt vs Richard H. Hunt), every case it touches would need manual review anyway, and it makes the method harder to describe. Flags keep the rule simple and leave the judgement to the reviewer.

**D8 – `award_in_scopus_pub_range = False` always means `review`.**
Prize-winning work precedes the prize, so a profile whose publications start after the award year cannot be the laureate's main profile. Example: the profile chosen for *Paul Greengard* (prize 2000, died 2019) has 17 documents published 2019–2026 — posthumous papers in a duplicate profile. The rule also catches profiles that stop long before the award; this produces some false alarms (laureates who stopped publishing before a late prize), which are quick to dismiss.

**D9 – The candidates file costs no extra requests.**
Citation metrics and publication years exist only in the Author Retrieval API, one request per profile, so they are retrieved for the chosen profile only. The candidates file holds what the searches already return (documents, affiliation, subject areas, ORCID, initials). An earlier version fetched publication years for the top candidates; it was dropped because of its cost.

**D10 – Access through the institutional network, no insttoken.**
The Author Search API requires a Scopus subscription. Requests from the institutional network are recognised by IP address; an insttoken that is not linked to the API key makes *every* request fail ("Institution Token is not associated with API Key"), so none is sent.

## 3. `match_status` and the review flags

### How the status is assigned

The status is assigned in two stages: the **choice** of a profile gives an initial status, then a **check** of the chosen profile may replace it by `review`.

**Before anything else – errors.** If the weekly Scopus quota is exhausted (HTTP 429) the run **stops**: that laureate and the following ones are not written (re-run later; finished requests are cached). Any other API or network error gives `error <type>: <message>` and the run goes on.

**Stage 1 – choice.** All candidates from all searches are merged and sorted by number of documents, and each is marked name-compatible or not (D4; candidates found by the ORCID search always count as compatible). The rules are tested **in this order**, and the first that applies decides:

| Order | Condition | Initial status | Profile chosen |
|---|---|---|---|
| 1 | a candidate **matches Wikidata**: its Scopus ID is in `wd_scopus_id`, or its ORCID in the search results is the `wd_orcid`, or it was found by the ORCID search | `likely` | among those, the one with most documents |
| 2 | no search found any candidate | `not found` | none |
| 3 | there are candidates, but none is name-compatible | `initials mismatch` | none |
| 4 | exactly one name-compatible candidate | `single` | that one |
| 5 | the name-compatible candidate with most documents has ≥ `DOMINANCE` × the documents of the second | `dominant` | the first |
| 6 | otherwise (two or more compatible candidates, no clear winner) | `ambiguous` | the first, all the same |

Notes: rule 1 does **not** apply the name check — Wikidata alone decides (D6), which is why the status is `likely` and not `confirmed`; with `ambiguous` a profile *is* chosen and its metrics filled in; with `not found` and `initials mismatch` nothing is chosen and the Scopus columns are empty.

**Stage 2 – check.** When a profile was chosen, it is retrieved (one Author Retrieval request), the flags are computed, and the status is **replaced by `review`** if any of these holds:

- `scopus_id_agrees` is `no` — Wikidata records a Scopus ID and the chosen profile is a different one;
- `orcid_agrees` is `no` — the chosen profile's ORCID differs from the Wikidata ORCID;
- `award_in_scopus_pub_range` is `False` — the award year lies outside the profile's publication range (D8).

Notes: any initial status can become `review`, including `likely` (e.g. Robert Edwards); an empty flag (nothing to compare: no Wikidata ID or ORCID, unknown publication years) never changes the status; `most_documents` never changes the status — it only informs the reviewer; `not found` and `initials mismatch` never become `review`, since there is no chosen profile to check.

### Status and flags at a glance

| `match_status` | Meaning | Suggested action |
|---|---|---|
| `likely` | the chosen profile has the Scopus ID or ORCID recorded in Wikidata | accept; spot-check, starting with `most_documents = False` |
| `dominant` | no Wikidata match; the chosen profile has ≥ `DOMINANCE` × the documents of the next compatible candidate | accept; spot-check |
| `single` | no Wikidata match; only one name-compatible candidate | accept; spot-check, especially if `n_candidates` is large |
| `ambiguous` | no Wikidata match and no clear winner | check the candidates |
| `initials mismatch` | profiles were found but none is name-compatible; nothing chosen | check the candidates |
| `review` | a profile was chosen, but it disagrees with Wikidata (`scopus_id_agrees` or `orcid_agrees` is `no`) or `award_in_scopus_pub_range` is `False` (stage 2) | check by hand |
| `not found` | no profile found by any search | check Scopus manually; may genuinely have no profile |
| `error …` | the API refused the request | fix access and re-run |

| Flag | `True` | `False` | Empty |
|---|---|---|---|
| `scopus_id_agrees` | chosen profile is one of the Scopus IDs in Wikidata | Wikidata points to another profile | Wikidata has no Scopus ID |
| `orcid_agrees` | Scopus and Wikidata ORCIDs are the same | they differ | either is missing |
| `most_documents` | chosen profile is the largest name-compatible candidate | a larger compatible profile exists (homonym or duplicate) | — |
| `award_in_scopus_pub_range` | `publication_first ≤ award_year ≤ publication_last` | award year outside the publication range | publication years unknown |

## 4. Output columns

### One row per laureate (`output/nobel_scopus_profiles.csv`)

Columns are grouped by what they let the reviewer compare; within each group the Wikidata value comes first (it is retrieved first), followed by the Scopus value. The last six columns summarise the checks.

| Group | Column | Meaning |
|---|---|---|
| Identification | `category`, `award_year`, `name` | from the input list |
| | `wd_qid` | Wikidata identifier of the person (e.g. `Q193652`; see `https://www.wikidata.org/wiki/<qid>`) |
| | `wd_label` | English name in Wikidata, used in the searches |
| | `wd_match` | how the name was matched to Wikidata: `award` (among the laureates of the same category and year — the reliable case), `exact`, `first+last`, `fuzzy`, `ambiguous` or `no match` |
| Identifiers | `wd_scopus_id` | Scopus Author ID(s) according to Wikidata (`;`-separated when several) |
| | `scopus_id` | Scopus Author ID of the chosen profile |
| | `scopus_url` | link to the chosen profile's Scopus page |
| | `wd_orcid` | ORCID according to Wikidata |
| | `orcid` | ORCID according to Scopus |
| Timeline | `wd_birth_year` | year of birth according to Wikidata |
| | `publication_first`, `publication_last` | years of the first and last publication in the Scopus profile |
| | `wd_death_year` | year of death according to Wikidata (empty if unknown or alive) — helps to read old affiliations and posthumous publications |
| Field | `wd_awards` | the person's Nobel Prizes according to Wikidata, e.g. `Nobel Prize in Chemistry (2001); Nobel Prize in Chemistry (2022)` |
| | `subject_areas` | Scopus subject areas of the profile with document counts, e.g. `BIOC (177); MULT (80)` |
| Affiliation | `wd_employers` | employers according to Wikidata (`;`-separated, no dates) |
| | `last_known_affiliation`, `last_known_affiliation_country` | parent institution and country of the affiliation on the most recent publications — not necessarily the current employer; for deceased laureates, the last one they published with |
| Production | `document_count` | documents in the profile |
| | `cited_by_count` | distinct documents citing the author |
| | `citation_count` | total citations (a document citing three of the author's papers counts three) |
| | `h_index`, `coauthor_count` | Scopus h-index and number of co-authors |
| Summary | `scopus_id_agrees`, `orcid_agrees`, `most_documents`, `award_in_scopus_pub_range` | review flags (section 3) |
| | `n_candidates` | number of distinct Scopus profiles found by all searches |
| | `match_status` | section 3 |

### One row per candidate (`output/nobel_scopus_candidates.csv`)

Up to `MAX_CANDIDATES` per laureate, most documents first, with only what the searches return:

| Column | Meaning |
|---|---|
| `category`, `award_year`, `name` | same keys as the profiles table |
| `found_by` | which searches found the candidate: `orcid`, `wd_label`, `wd_alias`, `name`, `fallback` |
| `rank` | position by number of documents |
| `scopus_id`, `scopus_name`, `initials` | the candidate's Scopus ID and name |
| `initials_match` | whether the candidate passed the name check (D4) |
| `documents` | number of documents |
| `affiliation`, `city`, `country` | current affiliation in Scopus |
| `subject_areas` | subject areas with document counts |
| `orcid`, `orcid_match` | ORCID in Scopus, and whether it equals the Wikidata ORCID |
| `is_wd_scopus_id` | whether this is the profile Wikidata points to |
| `scopus_url` | link to the profile |

## 5. How to review the output

1. Sort by `match_status` and handle, in this order: `review`, `ambiguous`, `initials mismatch`, `not found`.
2. For each, open the candidates of that laureate (last cell of this notebook) and their `scopus_url`. Signs of the right profile: subject areas matching the prize, affiliations known from the laureate's career, publications spanning the award year, many documents.
3. Two candidates with the same name and similar affiliations are usually **duplicate profiles of the same person** (their counts are split); different subject areas or affiliations mean **homonyms**.
4. Then spot-check `likely`, `dominant` and `single`, starting with rows where `most_documents` is `False` or `n_candidates` is large.
5. Record manual decisions in a separate column or file, so the automatic output can be regenerated without losing them.

## 6. Known limitations

- **Coverage**: Scopus indexes citations well from 1970 onwards; laureates whose work is older, and many Economics, Literature and Peace laureates, have small profiles or none.
- **Duplicate profiles**: Scopus may split one person into several profiles; the metrics then cover only the chosen one. The candidates file shows the others.
- **Wikidata quality**: Wikidata is crowd-sourced; its identifiers can be outdated or wrong (D6). Identity matching by prize and year is much more reliable than its Scopus IDs.
- **Name rules are heuristics**: unusual name forms (nicknames not recorded as aliases, very common names) can still defeat them; this is why every choice keeps its alternatives in the candidates file.
- **Aliases**: at most `MAX_ALIASES` Wikidata aliases are searched, taken in alphabetical order.
- **Affiliation**: `last_known_affiliation` is the affiliation on the latest publications, not the current employer. `wd_employers` is often incomplete and has no dates; laureates frequently move or become emeritus after the prize, so a difference between the two is not evidence of a wrong profile. For this reason no automatic affiliation (or subject-area) flag is computed: both were considered and rejected because a mismatch would mostly be a false alarm.
- **Snapshot**: all Scopus metrics change over time; they describe Scopus on the day the notebook ran.

## 7. Requirements and reproducibility

- Run on the institutional network (IP-based Scopus access) with `ELSEVIER_API_KEY` in a `.env` file and **no insttoken** (D10).
- `pip install pybliometrics python-dotenv pandas requests tqdm` (written for pybliometrics 4.4).
- Requests: one Wikidata query; about 2–4 Scopus searches and at most one profile retrieval per laureate — well within the weekly quotas of a standard key.
- Responses are cached (`cache/` for Wikidata, pybliometrics' own cache for Scopus), so re-running gives the same result without new requests; delete the caches to refresh. Record the run date printed at the end: it is the date of the Scopus snapshot.
- Read the output with `pd.read_csv(..., dtype=str)` to keep IDs as text.

In [ ]:
import hashlib
import json
import os
import re
import time
import unicodedata
from datetime import date
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd
import pybliometrics
import requests
from pybliometrics.exception import ScopusException, Scopus429Error
from pybliometrics.scopus import AuthorSearch, AuthorRetrieval
from requests.exceptions import RequestException
from tqdm.auto import tqdm

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

## 8. The code, step by step

### Configuration

| Parameter | Default | Role |
|---|---|---|
| `INPUT_FILE` | `data/nobel_laureates.csv` | input list with columns `category`, `award_year`, `name` |
| `OUTPUT_FILE`, `CANDIDATES_FILE` | `output/…` | the two output tables |
| `CACHE_DIR` | `cache` | cached Wikidata answers |
| `DOMINANCE` | 3 | the top candidate needs ≥ 3× the documents of the second to be `dominant` (D5) |
| `MAX_CANDIDATES` | 10 | candidates saved per laureate in the candidates file (`None` = all) |
| `MAX_ALIASES` | 5 | Wikidata aliases used as extra searches (D3) |
| `FUZZY_THRESHOLD` | 0.92 | whole-name similarity for the fallback Wikidata match (D2) |
| `TOKEN_THRESHOLD` | 0.75 | similarity for two name parts to count as the same, e.g. *Alfiorov* ≈ *Alferov* (D2) |

The cell also initialises pybliometrics with the API key only, so that no insttoken is sent (D10).

In [ ]:
INPUT_FILE = Path("data/nobel_laureates.csv")      # columns: category, award_year, name
OUTPUT_FILE = Path("output/nobel_scopus_profiles.csv")
CANDIDATES_FILE = Path("output/nobel_scopus_candidates.csv")
CACHE_DIR = Path("cache")          # Wikidata responses (pybliometrics caches Scopus on its own)

DOMINANCE = 3            # top candidate needs >= 3x the documents of the 2nd to count as "dominant"
MAX_CANDIDATES = 10      # candidates saved per laureate (most documents first); None = all
MAX_ALIASES = 5          # Wikidata alternative names used as extra searches
FUZZY_THRESHOLD = 0.92   # whole-name similarity for the Wikidata fallback match
TOKEN_THRESHOLD = 0.75   # similarity for two name parts to count as the same (Alfiorov ~ Alferov)

USER_AGENT = "nobel_oa/0.1 (https://github.com/gavieira/nobel_oa)"
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# Only the API key is passed: with explicit keys, pybliometrics ignores any InstToken saved in its
# config file. An insttoken not linked to the key makes every request fail, even on campus.
pybliometrics.init(keys=[os.environ["ELSEVIER_API_KEY"]])
try:
    from pybliometrics.utils.startup import get_insttokens
    assert not get_insttokens(), "An insttoken is configured; remove it to use IP-based access."
except ImportError:
    pass
print("pybliometrics", pybliometrics.__version__, "— API key set, no insttoken")

### Input list

`data/nobel_laureates.csv` holds the 257 laureates of 2000–2025 in Physics, Chemistry, Physiology or Medicine and Economics, as provided by the project (category, award year, name as written in the source list). The only edits made when building the file were removing a stray word ("Francês") after *Françoise Barré-Sinoussi* and collapsing double spaces. A person with two prizes (Sharpless, 2001 and 2022) has one row per prize.

In [ ]:
names = pd.read_csv(INPUT_FILE, dtype={"award_year": "Int64"})
names["name"] = names["name"].astype(str).str.split().str.join(" ")
names = names.drop_duplicates().reset_index(drop=True)   # Sharpless (2001, 2022) keeps one row per prize
print(len(names), "laureates")
names.head()

### Name helpers

Names from different sources are compared after normalisation: accents and punctuation removed, lower case, suffixes and titles (`Jr.`, `III`, `Sir`) dropped. For Scopus queries a name is split into given names and surname, keeping surname particles with the surname (`van der Waals`, `zur Hausen`).

In [ ]:
SUFFIXES = {"jr", "sr", "ii", "iii", "iv", "sir", "dr", "prof", "lord", "baron"}
PARTICLES = {"van", "von", "der", "den", "de", "da", "di", "du", "la", "le", "del", "dos", "das", "zur", "zu", "ten", "ter"}

def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def normalize_name(name):
    s = strip_accents(str(name)).lower()
    s = re.sub(r"[^a-z\s-]", " ", s).replace("-", " ")
    return " ".join(t for t in s.split() if t not in SUFFIXES)

def name_key(name):
    # first + last token, ignoring middle names
    tokens = normalize_name(name).split()
    return f"{tokens[0]} {tokens[-1]}" if len(tokens) > 1 else " ".join(tokens)

def split_name(name):
    # (given names, surname) for the Scopus query
    tokens = strip_accents(str(name)).replace(",", " ").split()
    tokens = [t for t in tokens if t.lower().strip(".") not in SUFFIXES]
    if len(tokens) == 1:
        return "", tokens[0]
    i = len(tokens) - 1
    while i > 1 and tokens[i - 1].lower() in PARTICLES:
        i -= 1
    return " ".join(tokens[:i]), " ".join(tokens[i:])

def similar(a, b):
    return SequenceMatcher(None, a, b).ratio()

### Step 1 – Wikidata

A single SPARQL query to the Wikidata Query Service returns every person (`P31 = Q5`, human) who received one of the six Nobel Prizes (`P166`, *award received*: Physics `Q38104`, Chemistry `Q44585`, Physiology or Medicine `Q80061`, Economics `Q47170`, Literature `Q37922`, Peace `Q35637`), with:

- the award year (`P585`, *point in time*, a qualifier of the award statement);
- the English label (or the multilingual label) and English aliases;
- Scopus Author IDs (`P1153`) and ORCIDs (`P496`), when recorded.

A second query returns the dates of birth (`P569`) and death (`P570`) and the employers (`P108`); it is separate because a person with many employers would otherwise multiply the rows of the first answer. Only the year of each date is kept.

Both answers are cached in `cache/wikidata/`, so later runs reuse them.

In [ ]:
def cached_get(url, params=None, headers=None, namespace="http", timeout=120):
    # GET with a file cache (one JSON file per URL + parameters) and retries on rate limits / server errors
    key = hashlib.sha1(json.dumps([url, params], sort_keys=True).encode()).hexdigest()
    path = CACHE_DIR / namespace / f"{key}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8")), True
    for attempt in range(5):
        r = requests.get(url, params=params, headers=headers, timeout=timeout)
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()
        data = r.json()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(data), encoding="utf-8")
        return data, False
    r.raise_for_status()

QUERY = '''
SELECT ?person ?label ?alias ?scopus ?orcid ?awardLabel ?date WHERE {
  # Physics, Chemistry, Physiology or Medicine, Economics, Literature, Peace
  VALUES ?award { wd:Q38104 wd:Q44585 wd:Q80061 wd:Q47170 wd:Q37922 wd:Q35637 }
  ?person p:P166 ?st .
  ?st ps:P166 ?award .
  OPTIONAL { ?st pq:P585 ?date }                # award year
  ?person wdt:P31 wd:Q5 .                       # people only
  ?person rdfs:label ?label .
  FILTER(LANG(?label) IN ("en", "mul"))
  OPTIONAL { ?person skos:altLabel ?alias . FILTER(LANG(?alias) = "en") }
  OPTIONAL { ?person wdt:P1153 ?scopus }        # Scopus Author ID
  OPTIONAL { ?person wdt:P496 ?orcid }          # ORCID
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". ?award rdfs:label ?awardLabel . }
}
'''

data, from_cache = cached_get("https://query.wikidata.org/sparql",
                              params={"query": QUERY, "format": "json"},
                              headers={"User-Agent": USER_AGENT, "Accept": "application/sparql-results+json"},
                              namespace="wikidata")
wd_raw = pd.DataFrame([{k: row.get(k, {}).get("value") for k in ["person", "label", "alias", "scopus", "orcid", "awardLabel", "date"]}
                       for row in data["results"]["bindings"]])
wd_raw["qid"] = wd_raw["person"].str.rsplit("/", n=1).str[-1]

CATEGORY_KEYWORDS = {"physics": "Physics", "chemistry": "Chemistry", "medicine": "Physiology or Medicine",
                     "econom": "Economics", "literature": "Literature", "peace": "Peace"}

def canonical_category(text):
    # "Nobel Prize in Physiology or Medicine" -> "Physiology or Medicine", matching the input list
    text = str(text).lower()
    return next((v for k, v in CATEGORY_KEYWORDS.items() if k in text), None)

def unique(values):
    return sorted({v for v in values if isinstance(v, str) and v})

# one row per (person, category, year): used to restrict the name matching to the right prize
awards = (wd_raw.assign(category=wd_raw["awardLabel"].map(canonical_category),
                        award_year=pd.to_numeric(wd_raw["date"].str[:4], errors="coerce").astype("Int64"))
          [["qid", "category", "award_year"]].drop_duplicates())
# one row per person: label, aliases, Scopus IDs, ORCIDs and the list of their Nobel Prizes
wikidata = (wd_raw.groupby("qid")
            .agg(wd_label=("label", "first"), wd_aliases=("alias", unique),
                 wd_scopus_ids=("scopus", unique), wd_orcids=("orcid", unique))
            .reset_index())
award_list = (wd_raw.assign(year=wd_raw["date"].str[:4])
              [["qid", "awardLabel", "year"]].drop_duplicates()
              .sort_values(["qid", "year"])
              .assign(text=lambda d: d["awardLabel"] + d["year"].map(lambda y: f" ({y})" if isinstance(y, str) else ""))
              .groupby("qid")["text"].agg("; ".join).rename("wd_awards").reset_index())
wikidata = wikidata.merge(award_list, on="qid", how="left")

# second query: birth/death dates and employers, kept separate because employers multiply the rows
LIFE_QUERY = '''
SELECT ?person ?birth ?death ?employerLabel WHERE {
  VALUES ?award { wd:Q38104 wd:Q44585 wd:Q80061 wd:Q47170 wd:Q37922 wd:Q35637 }
  ?person wdt:P166 ?award ;
          wdt:P31 wd:Q5 .
  OPTIONAL { ?person wdt:P569 ?birth }          # date of birth
  OPTIONAL { ?person wdt:P570 ?death }          # date of death
  OPTIONAL { ?person wdt:P108 ?employer }       # employer
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". ?employer rdfs:label ?employerLabel . }
}
'''
life_data, _ = cached_get("https://query.wikidata.org/sparql",
                          params={"query": LIFE_QUERY, "format": "json"},
                          headers={"User-Agent": USER_AGENT, "Accept": "application/sparql-results+json"},
                          namespace="wikidata")
life_raw = pd.DataFrame([{k: row.get(k, {}).get("value") for k in ["person", "birth", "death", "employerLabel"]}
                         for row in life_data["results"]["bindings"]],
                        columns=["person", "birth", "death", "employerLabel"])
life_raw["qid"] = life_raw["person"].str.rsplit("/", n=1).str[-1]
for col in ["birth", "death"]:
    life_raw[col] = pd.to_numeric(life_raw[col].str.extract(r"^(-?\d{4})")[0], errors="coerce")
life = (life_raw.groupby("qid")
        .agg(wd_birth_year=("birth", "min"), wd_death_year=("death", "min"),
             wd_employers=("employerLabel", lambda v: "; ".join(unique(v)) or None))
        .reset_index())
for col in ["wd_birth_year", "wd_death_year"]:
    life[col] = life[col].astype("Int64")
wikidata = wikidata.merge(life, on="qid", how="left")
if wikidata.empty:
    raise RuntimeError("The Wikidata query returned no laureates. Check that query.wikidata.org is reachable "
                       "from this machine; if an old/empty answer is cached, delete the cache/wikidata folder.")
print(f"{len(wikidata)} laureates in Wikidata ({'cached' if from_cache else 'fresh query'})")
print(awards.groupby("category")["qid"].nunique().to_string())

#### Matching the input names to Wikidata (D2)

1. **`award`** – among the laureates of the same category and year, the one whose name parts best match the input name. Each part of the input name scores its similarity to the closest part of each Wikidata name (label or alias) when it is at least `TOKEN_THRESHOLD`; the person with the highest total wins, and a tie is not accepted.
2. Fallbacks, used only if the category/year lookup finds nobody (e.g. a missing award year in Wikidata): **`exact`** – normalised name equal to a label or alias of exactly one person; **`first+last`** – same first and last name; **`fuzzy`** – whole-name similarity ≥ `FUZZY_THRESHOLD`. Otherwise **`ambiguous`** (several people) or **`no match`**.

The cell after the matching lists every name not matched by `award`, for inspection.

In [ ]:
# every label and alias of every laureate, normalised, for the name matching
name_forms = pd.concat([wd_raw[["qid", "label"]].rename(columns={"label": "form"}),
                        wd_raw[["qid", "alias"]].rename(columns={"alias": "form"})]).dropna().drop_duplicates()
name_forms["norm"] = name_forms["form"].map(normalize_name)
name_forms["key"] = name_forms["form"].map(name_key)
by_norm = name_forms.groupby("norm")["qid"].agg(set).to_dict()
by_key = name_forms.groupby("key")["qid"].agg(set).to_dict()
forms_by_qid = name_forms.groupby("qid")["norm"].agg(list).to_dict()

def token_score(name_tokens, form_tokens):
    # sum of similarities of the input name parts to their closest Wikidata name part (initials ignored)
    score = 0.0
    for t in name_tokens:
        if len(t) < 2:
            continue
        best = max((similar(t, f) for f in form_tokens), default=0.0)
        if best >= TOKEN_THRESHOLD:
            score += best
    return score

def match_by_award(name, category, year):
    # best-scoring laureate of the same category and year; None if nobody scores or there is a tie
    group = awards.loc[(awards["category"] == canonical_category(category)) & (awards["award_year"] == year), "qid"].unique()
    if len(group) == 0:
        return None
    tokens = normalize_name(name).split()
    ranked = sorted(((max(token_score(tokens, f.split()) for f in forms_by_qid.get(q, [""])), q) for q in group), reverse=True)
    if ranked[0][0] > 0 and (len(ranked) == 1 or ranked[0][0] > ranked[1][0]):
        return ranked[0][1], "award"
    return None

def match_by_name(name):
    # fallback: match across all laureates (exact, first+last, fuzzy)
    norm, key = normalize_name(name), name_key(name)
    if len(by_norm.get(norm, ())) == 1:
        return next(iter(by_norm[norm])), "exact"
    if len(by_key.get(key, ())) == 1:
        return next(iter(by_key[key])), "first+last"
    best = max(by_norm, key=lambda cand: similar(norm, cand), default=None)
    if best and similar(norm, best) >= FUZZY_THRESHOLD and len(by_norm[best]) == 1:
        return next(iter(by_norm[best])), "fuzzy"
    return None, ("ambiguous" if by_norm.get(norm) or by_key.get(key) else "no match")

def match_row(row):
    if pd.notna(row.get("category")) and pd.notna(row.get("award_year")):
        m = match_by_award(row["name"], row["category"], row["award_year"])
        if m:
            return m
    return match_by_name(row["name"])

m = names.apply(match_row, axis=1)
names["wd_qid"], names["wd_match"] = m.str[0], m.str[1]
laureates = names.merge(wikidata, left_on="wd_qid", right_on="qid", how="left").drop(columns="qid")
for col in ["wd_aliases", "wd_scopus_ids", "wd_orcids"]:
    laureates[col] = laureates[col].apply(lambda v: v if isinstance(v, list) else [])
matched = laureates["wd_qid"].notna().sum()
print(f"{matched} of {len(laureates)} names matched to Wikidata")
if matched == 0:
    print("⚠️  No name matched: check that the input columns are named category, award_year and name.")
laureates["wd_match"].value_counts()

In [ ]:
# names not matched among the laureates of their category and year — worth a look
laureates.loc[laureates["wd_match"] != "award", ["category", "award_year", "name", "wd_match", "wd_label", "wd_qid"]]

### Step 2 – Scopus candidates (D3, D4)

For each laureate the searches are, in this order (identical queries are sent only once):

1. `ORCID(...)` for each ORCID in Wikidata;
2. `AUTHLAST(surname) AND AUTHFIRST(first given name)` for the Wikidata name, up to `MAX_ALIASES` Wikidata aliases (e.g. *Tim Hunt*), and the name in the input list.

Only if none of these finds a name-compatible candidate, fallbacks are tried: first initial only, each other given name (for people known by a middle name, e.g. *Kamer **Daron** Acemoğlu*), and the reversed order for two-word names written surname first (*Tu Youyou*).

**Name check.** A candidate is compatible only if (1) its Scopus surname is the laureate's surname — Scopus sometimes returns profiles whose *other* names match, such as *Axel Richard Stocker* for `AUTHLAST(Axel)`; (2) its first initial is one of the laureate's given-name initials; and (3) when the laureate's middle names are known, all its initials belong to them — *Richard **H.** Hunt* is ruled out for *Richard **Timothy** Hunt*, while *R. A. Axel* is accepted for *Richard Axel*, whose middle name is not in the list. All the laureate's names above are used. Candidates found by ORCID are always compatible.

In [ ]:
def name_query(name):
    # main Scopus query for one name form: full surname + first given name (D3)
    given, surname = split_name(name)
    last = surname.replace('"', "")
    if not given:
        return None
    first = given.split()[0].replace('"', "").rstrip(".")
    return f'AUTHLAST("{last}") AND AUTHFIRST("{first}")'

def fallback_queries(name):
    # extra queries, used only when the main ones find no compatible candidate (D3)
    given, surname = split_name(name)
    last, parts = surname.replace('"', ""), [p.replace('"', "").rstrip(".") for p in given.split()]
    if not parts:
        return []
    q = [f'AUTHLAST("{last}") AND AUTHFIRST("{parts[0][0]}")']
    q += [f'AUTHLAST("{last}") AND AUTHFIRST("{p}")' for p in parts[1:] if len(p) > 1]
    if len(parts) == 1 and " " not in last:
        q.append(f'AUTHLAST("{parts[0]}") AND AUTHFIRST("{last}")')   # surname written first
    return q

def name_forms_for(row):
    # (source, name) pairs: Wikidata name, Wikidata aliases, name in the input list; duplicates removed
    forms = [("wd_label", row["wd_label"])] if isinstance(row.get("wd_label"), str) else []
    forms += [("wd_alias", a) for a in row["wd_aliases"][:MAX_ALIASES]]
    forms.append(("name", row["name"]))
    seen, out = set(), []
    for source, form in forms:
        if normalize_name(form) not in seen:
            seen.add(normalize_name(form))
            out.append((source, form))
    return out

def laureate_names(row):
    return [form for _, form in name_forms_for(row)]

def search_plan(row):
    # (source, query) pairs; identical queries are sent only once
    plan = [("orcid", f"ORCID({o})") for o in row["wd_orcids"]]
    plan += [(source, name_query(form)) for source, form in name_forms_for(row) if name_query(form)]
    seen, out = set(), []
    for source, q in plan:
        if q not in seen:
            seen.add(q)
            out.append((source, q))
    return out

def possible_surnames(form):
    # surname of a name form; for two-word names also the first word (names written surname first, e.g. Tu Youyou)
    tokens = normalize_name(form).split()
    out = [normalize_name(split_name(form)[1]).split()]
    if len(tokens) == 2:
        out.append(tokens[:1])
    return out

def same_surname(a, b):
    return bool(a and b) and (similar(" ".join(a), " ".join(b)) >= 0.85 or a[-1] == b[-1])

def initials_match(cand, forms):
    # Is this Scopus profile compatible with the laureate's names?
    # 1. its surname must be the laureate's surname (Axel Richard *Stocker* is not Richard *Axel*);
    # 2. its first initial must be one of the laureate's given-name initials;
    # 3. extra initials only count against it when the laureate's middle names are known:
    #    Richard *H.* Hunt is ruled out for Richard *Timothy* Hunt, but R. A. Axel is fine for Richard Axel.
    cand_surname = normalize_name(cand.surname or "").split()
    given_initials, middle_known = set(), False
    for form in forms:
        tokens = normalize_name(form).split()
        for surname in possible_surnames(form):
            if same_surname(surname, cand_surname):
                given = [t for t in tokens if t not in surname and t not in PARTICLES]
                given_initials |= {t[0] for t in given}
                middle_known |= len(given) >= 2
    if not given_initials:
        return False                                     # surname does not match any name form
    if cand.initials:
        # one letter per initial: "Zh.I." -> z, i; "J.-P." -> j, p
        cand_initials = [chunk[0] for chunk in re.split(r"[^a-z]+", strip_accents(cand.initials).lower()) if chunk]
    else:
        cand_initials = [t[0] for t in normalize_name(cand.givenname or "").split()]
    if not cand_initials:
        return True
    if cand_initials[0] not in given_initials:
        return False
    return not middle_known or set(cand_initials) <= given_initials

def collect_candidates(row):
    # run the searches, merge their results by Scopus ID, apply the name check;
    # returns [(scopus_id, {"author": ..., "found_by": [...], "initials_match": bool})] sorted by documents
    forms = laureate_names(row)
    found = {}                      # scopus_id -> {"author": Author, "found_by": [...]}
    def run(source, query):
        s = AuthorSearch(query, refresh=False)
        for a in s.authors or []:
            cid = a.eid.split("-")[-1]
            entry = found.setdefault(cid, {"author": a, "found_by": []})
            if source not in entry["found_by"]:
                entry["found_by"].append(source)
    for source, query in search_plan(row):
        run(source, query)
    def any_compatible():
        return any("orcid" in e["found_by"] or initials_match(e["author"], forms) for e in found.values())
    if not any_compatible():
        tried = {q for _, q in search_plan(row)}
        for form in forms:
            for q in fallback_queries(form):
                if q not in tried:
                    tried.add(q)
                    run("fallback", q)
            if any_compatible():
                break
    for cid, e in found.items():
        e["initials_match"] = "orcid" in e["found_by"] or initials_match(e["author"], forms)
    return sorted(found.items(), key=lambda kv: int(kv[1]["author"].documents or 0), reverse=True)

### Step 3 – Choice and checks (D5–D8)

`choose()` implements stage 1 of section 3 (rules 1–6, in that order); the stage-2 check is applied in the run loop below, after the chosen profile is retrieved. In short:

1. `likely` — a candidate has the Scopus ID or the ORCID that Wikidata gives (or was found by the Wikidata ORCID); no name check;
2. `not found` — no candidate at all;
3. `initials mismatch` — candidates, but none name-compatible; nothing chosen;
4. `single`, 5. `dominant`, 6. `ambiguous` — among the name-compatible candidates, sorted by documents;
7. then `review` replaces any of the statuses with a chosen profile when `scopus_id_agrees` or `orcid_agrees` is `no`, or `award_in_scopus_pub_range` is `False`.

`most_documents` does not change the status: it is a flag for the reviewer (e.g. a `likely` profile with `most_documents = False` may be a small duplicate of a larger profile, as for Robert Edwards).

The chosen profile is then retrieved with the Author Retrieval API (`ENHANCED` view: one request), which provides the metrics, the publication range and the affiliation.

In [ ]:
def scopus_url(author_id):
    return f"https://www.scopus.com/authid/detail.uri?authorId={author_id}"

def join_unique(values):
    vals = list(dict.fromkeys(v for v in values if v))
    return "; ".join(vals) or None

def agrees(value, reference):
    # "yes"/"no" comparison with the Wikidata values; None when there is nothing to compare
    if not value or not reference:
        return None
    return "yes" if value in reference else "no"

def choose(row, cands):
    # D5: the candidate Wikidata points to (Scopus ID or ORCID); otherwise the compatible one with most documents
    wd_ids, wd_orcids = set(row["wd_scopus_ids"]), set(row["wd_orcids"])
    confirmed = [(cid, e) for cid, e in cands
                 if cid in wd_ids or (e["author"].orcid and e["author"].orcid in wd_orcids) or "orcid" in e["found_by"]]
    if confirmed:
        return confirmed[0][0], "likely"
    compatible = [(cid, e) for cid, e in cands if e["initials_match"]]
    if not cands:
        return None, "not found"
    if not compatible:
        return None, "initials mismatch"
    if len(compatible) == 1:
        return compatible[0][0], "single"
    top, second = (int(e["author"].documents or 0) for _, e in compatible[:2])
    return compatible[0][0], ("dominant" if top >= DOMINANCE * max(second, 1) else "ambiguous")

def profile_fields(author_id):
    # one Author Retrieval request for the chosen profile: metrics, publication range, affiliation
    au = AuthorRetrieval(author_id, view="ENHANCED", refresh=False)
    affs = au.affiliation_current or []
    first, last = au.publication_range or (None, None)
    return {
        "scopus_id": str(au.identifier), "orcid": au.orcid,
        "document_count": au.document_count, "cited_by_count": au.cited_by_count,
        "citation_count": au.citation_count, "h_index": au.h_index, "coauthor_count": au.coauthor_count,
        "publication_first": first, "publication_last": last,
        "last_known_affiliation": join_unique(a.parent_preferred_name or a.preferred_name for a in affs),
        "last_known_affiliation_country": join_unique(a.country for a in affs),
        "scopus_url": scopus_url(au.identifier),
    }

### Run

For each laureate: collect candidates, choose, retrieve the chosen profile, compute the flags, and write one row to each table. If the weekly Scopus quota runs out (HTTP 429) the loop stops; everything already fetched is cached, so re-running later continues where it stopped. Other API errors are recorded in `match_status` and the loop goes on. Both output files are overwritten on every run (written to a temporary file first and then swapped in, which also works when the old file was created by another user, e.g. a previous container).

In [ ]:
COLUMNS = [
    # identification
    "category", "award_year", "name", "wd_qid", "wd_label", "wd_match",
    # identifiers: Wikidata first, then the chosen Scopus profile
    "wd_scopus_id", "scopus_id", "scopus_url", "wd_orcid", "orcid",
    # timeline, in chronological order: compare with award_year
    "wd_birth_year", "publication_first", "publication_last", "wd_death_year",
    # field
    "wd_awards", "subject_areas",
    # affiliation
    "wd_employers", "last_known_affiliation", "last_known_affiliation_country",
    # production
    "document_count", "cited_by_count", "citation_count", "h_index", "coauthor_count",
    # summary: agreement flags, sanity flags and status
    "scopus_id_agrees", "orcid_agrees", "most_documents", "award_in_scopus_pub_range", "n_candidates", "match_status",
]

def save_csv(df, path):
    # write to a temporary file, then replace the old one: works even if the old file belongs to another
    # user (e.g. created by a previous container), since replacing only needs write access to the folder
    tmp = Path(path).with_name(Path(path).name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

rows, candidate_rows = [], []
for _, lau in tqdm(laureates.iterrows(), total=len(laureates)):
    row = {k: lau.get(k) for k in ["category", "award_year", "name", "wd_qid", "wd_label", "wd_birth_year",
                                   "wd_death_year", "wd_match", "wd_awards", "wd_employers"]}
    row["wd_orcid"] = "; ".join(lau["wd_orcids"]) or None
    row["wd_scopus_id"] = "; ".join(lau["wd_scopus_ids"]) or None
    try:
        cands = collect_candidates(lau)
        chosen, status = choose(lau, cands)
        row.update({"n_candidates": len(cands), "match_status": status})
        if chosen:
            row.update(profile_fields(chosen))
            row["subject_areas"] = dict(cands)[chosen]["author"].areas or None   # from the search, no extra request
            row["scopus_id_agrees"] = agrees(row["scopus_id"], lau["wd_scopus_ids"])
            row["orcid_agrees"] = agrees(row["orcid"], lau["wd_orcids"])
            # sanity flags for the reviewer (search document counts, so all candidates are compared alike)
            docs = {cid: int(e["author"].documents or 0) for cid, e in cands if e["initials_match"] or cid == chosen}
            row["most_documents"] = docs[chosen] >= max(docs.values())
            first, last, year = row["publication_first"], row["publication_last"], lau.get("award_year")
            row["award_in_scopus_pub_range"] = (first <= year <= last) if None not in (first, last) and pd.notna(year) else None
            # D6/D8: disagreement with Wikidata or publications not covering the award year -> review
            if "no" in (row["scopus_id_agrees"], row["orcid_agrees"]) or row["award_in_scopus_pub_range"] is False:
                row["match_status"] = "review"
        for rank, (cid, e) in enumerate(cands[:MAX_CANDIDATES], 1):
            a = e["author"]
            candidate_rows.append({
                "category": lau.get("category"), "award_year": lau.get("award_year"), "name": lau["name"],
                "found_by": "; ".join(e["found_by"]),
                "rank": rank, "scopus_id": cid, "scopus_name": f"{a.givenname or ''} {a.surname or ''}".strip(),
                "initials": a.initials, "initials_match": e["initials_match"], "documents": a.documents,
                "affiliation": a.affiliation, "city": a.city, "country": a.country, "subject_areas": a.areas,
                "orcid": a.orcid, "orcid_match": agrees(a.orcid, lau["wd_orcids"]),
                "is_wd_scopus_id": cid in lau["wd_scopus_ids"], "scopus_url": scopus_url(cid)})
    except Scopus429Error:
        print("Weekly quota exhausted — stopping. Re-run later: finished searches are cached.")
        break
    except (ScopusException, RequestException) as e:
        row.update({"match_status": f"error {type(e).__name__}: {e}"[:200]})
    rows.append(row)

profiles = pd.DataFrame(rows).reindex(columns=COLUMNS)
INT_FIELDS = ["wd_birth_year", "wd_death_year", "document_count", "cited_by_count", "citation_count", "h_index",
              "coauthor_count", "publication_first", "publication_last", "n_candidates"]
profiles[INT_FIELDS] = profiles[INT_FIELDS].apply(pd.to_numeric, errors="coerce").astype("Int64")
candidates = pd.DataFrame(candidate_rows)
if not candidates.empty:
    for col in ["rank", "documents"]:
        candidates[col] = pd.to_numeric(candidates[col], errors="coerce").astype("Int64")

save_csv(profiles, OUTPUT_FILE)        # overwrites the previous output
save_csv(candidates, CANDIDATES_FILE)
print(f"Saved {OUTPUT_FILE} and {CANDIDATES_FILE} ({date.today()})")
print("Filled Wikidata fields:", profiles[["wd_qid", "wd_label", "wd_birth_year", "wd_death_year", "wd_employers",
                                         "wd_scopus_id", "wd_orcid"]].notna().sum().to_dict())
print("Read them back with pd.read_csv(..., dtype=str) to keep the IDs as text.")
profiles["match_status"].value_counts()

### Uncertain cases

The first table lists the rows that need a manual look (every status except `likely`, `dominant` and `single`); the second shows their top five candidates. See section 5 for how to decide.

In [ ]:
uncertain = profiles.loc[~profiles["match_status"].isin(["likely", "single", "dominant"]),
                         ["category", "award_year", "name", "wd_label", "match_status", "n_candidates",
                          "scopus_id", "scopus_id_agrees", "orcid_agrees", "most_documents", "award_in_scopus_pub_range", "scopus_url"]]
uncertain

In [ ]:
candidates[candidates["name"].isin(uncertain["name"]) & (candidates["rank"] <= 5)]